In [16]:
from pathlib import Path
import unicodedata

import pandas as pd
from rapidfuzz import fuzz

base_path = Path.cwd()
s1 = pd.read_excel(base_path / 's1.xlsx')
s2 = pd.read_excel(base_path / 's2.xlsx')

TITLE_SIMILARITY_THRESHOLD = 90
AUTHORS_SIMILARITY_THRESHOLD = 90


def normalize_text(value):
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKD', str(value))
    text = ''.join(char for char in text if not unicodedata.combining(char))
    return text.lower().strip()

In [17]:
# Merge both sources and remove exact duplicates by DOI.
unified_df = pd.concat(
    [s1.assign(source='s1'), s2.assign(source='s2')],
    ignore_index=True,
)

records_s1 = len(s1)
records_s2 = len(s2)
records_unified = len(unified_df)

normalized_doi = (
    unified_df['doi']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

has_doi = normalized_doi.ne('') & normalized_doi.ne('nan')
duplicate_doi_count = int(normalized_doi[has_doi].duplicated(keep='first').sum())

doi_filtered_df = unified_df.loc[~has_doi | ~normalized_doi.duplicated(keep='first')].copy()

In [18]:
# Check duplicates after DOI filtering using both title and author similarity.
doi_filtered_df['title_normalized'] = doi_filtered_df['title'].map(normalize_text)
doi_filtered_df['authors_normalized'] = doi_filtered_df['author_names'].map(normalize_text)

seen_records = []
keep_mask = []

for title, authors in zip(
    doi_filtered_df['title_normalized'],
    doi_filtered_df['authors_normalized'],
):
    if not title or not authors:
        keep_mask.append(True)
        continue

    is_duplicate = any(
        fuzz.token_sort_ratio(title, seen_title) >= TITLE_SIMILARITY_THRESHOLD
        and fuzz.token_sort_ratio(authors, seen_authors) >= AUTHORS_SIMILARITY_THRESHOLD
        for seen_title, seen_authors in seen_records
    )
    keep_mask.append(not is_duplicate)

    if not is_duplicate:
        seen_records.append((title, authors))

duplicate_title_author_count = int((~pd.Series(keep_mask, index=doi_filtered_df.index)).sum())

clean_df = doi_filtered_df.loc[keep_mask].copy()
clean_df.drop(columns=['title_normalized', 'authors_normalized'], inplace=True)
clean_df.reset_index(drop=True, inplace=True)
final_records = len(clean_df)

summary_df = pd.DataFrame(
    {
        'Step': [
            'Records from s1 file',
            'Records from s2 file',
            'Initial merge s1 + s2',
            'Duplicates detected by DOI',
            f'Duplicates detected by similar title and authors (>= {TITLE_SIMILARITY_THRESHOLD}% and >= {AUTHORS_SIMILARITY_THRESHOLD}%)',
            'Final records without duplicates',
        ],
        'Value': [
            records_s1,
            records_s2,
            records_unified,
            duplicate_doi_count,
            duplicate_title_author_count,
            final_records,
        ],
    }
)

display(summary_df)
print(
    'Similarity check applied with RapidFuzz token_sort_ratio and thresholds '
    f'{TITLE_SIMILARITY_THRESHOLD}% title / {AUTHORS_SIMILARITY_THRESHOLD}% authors.'
)

,Step,Value
0,Records from s1 file,1383
1,Records from s2 file,131
2,Initial merge s1 + s2,1514
3,Duplicates detected by DOI,70
4,Duplicates detected by similar title and autho...,3
5,Final records without duplicates,1441


Similarity check applied with RapidFuzz token_sort_ratio and thresholds 90% title / 90% authors.


In [ ]:
# Export the cleaned dataset.
output_path = base_path / 'salida' / 'results.csv'
clean_df.to_csv(output_path, index=False)
print(f'CSV saved to: {output_path}')
display(clean_df.head())

CSV saved to: /home/d4ndres/Escritorio/revision sistematica/salida/resultado_unificado_sin_duplicados.csv


,eid,doi,title,description,publicationName,coverDate,citedby_count,aggregationType,subtypeDescription,affiliation_country,...,issn,eIssn,pub_year,citescore,citescore_percentile,citescore_quartile,sjr,snip,string_search,source
0,2-s2.0-105040647892,10.1016/j.eswa.2026.132978,Scene-level markerless registration for indust...,In large-scale knowledge-intensive manual asse...,Expert Systems with Applications,2026-11-01,0,Journal,Article,China,...,09574174,NaN,2026,14.5,97.0,Q1,1.939,2.421,"TITLE-ABS-KEY (""manual assembl*"" OR ""industria...",s1
1,2-s2.0-105038751944,10.1115/1.4071655,Representation Learning of Problem-Solving Beh...,The number of manufacturing jobs in the US has...,Journal of Mechanical Design,2026-11-01,0,Journal,Article,United States;Canada;United States;United States,...,10500472,NaN,2026,5.4,79.0,Q1,0.821,1.281,"TITLE-ABS-KEY (""manual assembl*"" OR ""industria...",s1
2,2-s2.0-105044413202,10.1016/j.jmsy.2026.06.021,Vision language model-enhanced embodied intell...,Human-robot collaborative (HRC) assembly is pi...,Journal of Manufacturing Systems,2026-10-01,0,Journal,Article,China;China;China,...,02786125,NaN,2026,21.6,99.0,Q1,3.304,3.747,"TITLE-ABS-KEY (""manual assembl*"" OR ""industria...",s1
3,2-s2.0-105041368093,10.1016/j.robot.2026.105561,Uncertainty-aware monocular 6DoF pose tracking...,Augmented reality (AR) has been widely regarde...,Robotics and Autonomous Systems,2026-10-01,0,Journal,Article,China;China;China,...,09218890,NaN,2026,7.6,97.0,Q1,1.436,1.964,"TITLE-ABS-KEY (""manual assembl*"" OR ""industria...",s1
4,2-s2.0-105043778683,10.1007/s10055-026-01385-4,Smart adaptive augmented reality SAARTool for ...,Enhancing technical training through adaptive ...,Virtual Reality,2026-09-01,0,Journal,Article,Spain;Spain,...,13594338,14349957,2026,10.6,90.0,Q1,1.037,1.917,"TITLE-ABS-KEY (""manual assembl*"" OR ""industria...",s1
